# 01 — Load & Explore the MCAS Compound Library

> ⚠️ **Not medical advice.** Research/hypothesis use only. See `docs/disclaimers.md`.

What this notebook does:
1. Loads `data/compounds/MCAS_Compound_Library_v1.csv` produced by `scripts/build_compound_library.py`.
2. Renders a 2D molecule grid grouped by category (rescue / maintenance / remission).
3. Enumerates a few PyTDC datasets we can use downstream as ADMET / safety proxies.

Runs on Mac CPU. No GPU needed.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw, AllChem

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CSV_PATH = REPO_ROOT / 'data' / 'compounds' / 'MCAS_Compound_Library_v1.csv'
print('Library:', CSV_PATH)

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f'{len(df)} compounds total')
df['category'].value_counts()

In [ ]:
small_molecules = df[df['smiles'].notna() & (df['smiles'] != '')].copy()
biologics = df[df['smiles'].isna() | (df['smiles'] == '')]
print('Small molecules:', len(small_molecules))
print('Biologics / extracts:', len(biologics))
biologics[['name', 'category', 'biologic_flag']]

In [ ]:
def grid(category, per_row=4, sub_image=(260, 220)):
    sub = small_molecules[small_molecules['category'] == category].head(20)
    mols = [Chem.MolFromSmiles(s) for s in sub['canonical_smiles']]
    legends = list(sub['name'])
    return Draw.MolsToGridImage(mols, molsPerRow=per_row, subImgSize=sub_image, legends=legends)

grid('rescue')

In [ ]:
grid('maintenance')

In [ ]:
grid('remission')

In [ ]:
from rdkit.Chem import Descriptors, Lipinski
rows = []
for _, r in small_molecules.iterrows():
    mol = Chem.MolFromSmiles(r['canonical_smiles'])
    if mol is None:
        continue
    rows.append({
        'name': r['name'],
        'category': r['category'],
        'mw': Descriptors.MolWt(mol),
        'logp': Descriptors.MolLogP(mol),
        'hbd': Lipinski.NumHDonors(mol),
        'hba': Lipinski.NumHAcceptors(mol),
        'rotb': Lipinski.NumRotatableBonds(mol),
        'tpsa': Descriptors.TPSA(mol),
        'qed': Descriptors.qed(mol),
    })
props = pd.DataFrame(rows)
props.sort_values('qed', ascending=False).head(15)

## PyTDC datasets we can use as ADMET / safety proxies

Mast-cell-specific bioassay data is sparse and scattered across ChEMBL/PubChem. As a starting point we use PyTDC's curated ADMET / Tox tasks to train models that predict properties relevant to MCAS lead candidates (hERG cardiac risk, CYP DDIs, BBB penetration, etc.).

Install:
```bash
pip install PyTDC
```

In [ ]:
try:
    from tdc.single_pred import ADME, Tox
    print('ADME tasks: ADME.dataset_names')
    print('Tox tasks: Tox.dataset_names')
    # Example: hERG cardiac liability (relevant for mast cell drugs that often hit hERG)
    herg = Tox(name='hERG')
    print('hERG dataset:', len(herg.get_data()), 'rows')
except ImportError:
    print('PyTDC not installed. Run: pip install PyTDC')

## Next

- `02_qsar_deepchem.ipynb` — train a QSAR model on a PyTDC task and score this library.
- `03_reinvent_generation_colab.ipynb` — generate novel analogs seeded from quercetin / luteolin / sulforaphane on Colab GPU.
- `04_virtual_screening.ipynb` — dock the library + generated SMILES against MRGPRX2 / KIT / FcεRI.
- `05_hypothesis_ranking.ipynb` — join QSAR + docking + evidence → ranked rescue / maintenance / remission candidates.